In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import matplotlib.pyplot as plt

file_path = "/content/drive/MyDrive/부산_BigData_공모전/02_processed_data/유휴공유재산_1차정제.csv"

df = pd.read_csv(file_path)

print(df.shape)
df.head()

In [ ]:
# 구·군별 재산 개수와 총면적 확인

gu_summary = (
    df.groupby("gugun")
      .agg(
          property_count=("lctnAddr", "count"),
          total_area=("zoneArea", "sum"),
          median_area=("zoneArea", "median")
      )
      .sort_values("property_count", ascending=False)
)

gu_summary

In [ ]:
# 지목별 재산 개수와 총면적 확인

land_summary = (
    df.groupby("land_type")
      .agg(
          property_count=("lctnAddr", "count"),
          total_area=("zoneArea", "sum"),
          median_area=("zoneArea", "median")
      )
      .sort_values("property_count", ascending=False)
)

land_summary

In [ ]:
# 활용 규모를 보기 위해 면적 구간 생성

bins = [-1, 1, 10, 30, 50, 100, 300, 1000, float("inf")]
labels = [
    "1㎡ 이하",
    "1~10㎡",
    "10~30㎡",
    "30~50㎡",
    "50~100㎡",
    "100~300㎡",
    "300~1000㎡",
    "1000㎡ 초과"
]

df["area_group"] = pd.cut(
    df["zoneArea"],
    bins=bins,
    labels=labels
)

area_summary = (
    df["area_group"]
    .value_counts()
    .sort_index()
)

area_summary

In [ ]:
# 지목별로 면적 구간이 어떻게 구성되는지 확인

land_area_table = pd.crosstab(
    df["land_type"],
    df["area_group"]
)

land_area_table["전체"] = land_area_table.sum(axis=1)

land_area_table.sort_values("전체", ascending=False)

In [ ]:
# 구·군별 지목 구성을 확인해 단순 재산 건수 차이의 원인을 파악

gu_land_table = pd.crosstab(
    df["gugun"],
    df["land_type"]
)

gu_land_table["전체"] = gu_land_table.sum(axis=1)

gu_land_table.sort_values("전체", ascending=False)

# 유휴 공유재산 기초 EDA

## 주요 결과

정제된 유휴 공유재산 9,404건의 구·군, 지목, 면적 분포를 확인하였다.

### 구·군별 특징

- 부산진구가 4,793건으로 가장 많음
- 영도구 1,780건, 중구 743건, 동구 522건 순
- 그러나 구·군별 지목 구성이 크게 달라 단순 건수 비교에는 주의가 필요함

특히 부산진구 4,793건 중 도로가 3,223건으로 큰 비중을 차지한다.

따라서 부산진구의 유휴 공유재산 건수가 많다는 사실을
곧바로 활용 가능한 재산이 많다는 의미로 해석하지 않는다.

### 지목별 특징

- 대: 4,836건
- 도로: 3,296건
- 전: 337건
- 임야: 316건
- 답: 199건
- 기타 지목 존재

전체 데이터에서 도로 지목이 상당한 비중을 차지한다.

### 면적별 특징

- 1㎡ 이하: 603건
- 1~10㎡: 2,292건
- 10~30㎡: 1,979건
- 30~50㎡: 1,238건
- 50~100㎡: 1,369건
- 100~300㎡: 1,126건
- 300~1000㎡: 530건
- 1000㎡ 초과: 267건

10㎡ 이하 재산은 총 2,895건으로,
전체 데이터에 극소면적 필지가 상당수 존재한다.

## 분석 판단

단순 재산 건수만으로 활용 우선지역을 선정하지 않는다.

또한 `도로 제외`, `10㎡ 이하 제외`와 같은 일률적인 기준을
현재 단계에서 적용하지 않는다.

면적이 작은 재산도 활용유형에 따라 소규모 생활공간으로 활용될 가능성이 있으며,
반대로 면적이 큰 재산도 지목 및 물리적 조건에 따라 활용이 어려울 수 있기 때문이다.

따라서 다음 순서로 분석한다.

생활수요 분석
→ 기존 공공서비스 공급 분석
→ 수요-공급 불균형 분석
→ 필요한 활용유형 결정
→ 활용유형별 지목·면적 조건 설정
→ 유휴 공유재산 후보 선별



# 16개 구·군 유휴 공유재산 데이터 보완

유휴 공유재산 API 조회 결과 남구와 북구에서 NODATA가 발생하여
두 지역의 유휴 공유재산 현황을 API만으로 확인할 수 없었다.

기존 교차검증 과정에서 구·군 공식 공유재산 자료와 API 자료의
주소·지목·면적 등 핵심 항목이 높은 수준으로 일치함을 확인하였다.

이에 남구와 북구는 각 구에서 공개한 2026년 공유재산 현황 자료를
보완자료로 활용하여 기존 API 기반 데이터와 통합한다.

- 기존 14개 구·군: 2026년 5월 유휴 공유재산 API
- 남구: 공유재산 현황 (2026.02.11)
- 북구: 공유재산 현황 (2026.06.30)

단, 남구 공개자료에는 지목 정보가 제공되지 않으므로
해당 값은 임의로 추정하지 않고 결측값으로 유지한다.

또한 데이터 출처와 기준시점의 차이를 추적할 수 있도록
통합 데이터에 `data_source`와 `reference_date`를 별도로 기록한다.

In [ ]:
# 남구·북구 파일은 인코딩 방식이 서로 달라 각각 지정해서 불러온다.

import pandas as pd

namgu_path = "/content/drive/MyDrive/부산_BigData_공모전/01_raw_data/유휴공유재산/부산광역시 남구_공유재산 현황_20260211.csv"
bukgu_path = "/content/drive/MyDrive/부산_BigData_공모전/01_raw_data/유휴공유재산/부산광역시 북구_공유재산 현황_20260630.csv"

namgu = pd.read_csv(namgu_path, encoding="euc-kr")
bukgu = pd.read_csv(bukgu_path, encoding="utf-8-sig")

print("남구:", namgu.shape)
print(namgu.columns.tolist())

print("\n북구:", bukgu.shape)
print(bukgu.columns.tolist())

In [ ]:
# 남구·북구 공식 공개자료를 기존 유휴 공유재산 데이터 구조에 맞춘다.
# 이 단계에서는 행을 삭제하거나 임의로 값을 채우지 않는다.

namgu_std = pd.DataFrame({
    "gugun": "남구",
    "insttCode": pd.NA,
    "proyUsgNm": pd.NA,
    "lctnAddr": namgu["소재지"].astype(str).str.strip(),
    "landLdcgNm": pd.NA,
    "zoneArea": pd.to_numeric(namgu["면적(제곱미터)"], errors="coerce"),
    "referenceDate": "2026-02-11",
    "land_type": pd.NA,
    "data_source": "남구 공유재산 현황(공식 보완자료)"
})

bukgu_std = pd.DataFrame({
    "gugun": "북구",
    "insttCode": pd.NA,
    "proyUsgNm": pd.NA,
    "lctnAddr": bukgu["소재지"].astype(str).str.strip(),
    "landLdcgNm": bukgu["지목"].astype(str).str.strip(),
    "zoneArea": pd.to_numeric(bukgu["단면적"], errors="coerce"),
    "referenceDate": "2026-06-30",
    "land_type": bukgu["지목"].astype(str).str.strip(),
    "data_source": "북구 공유재산 현황(공식 보완자료)"
})

print("남구 표준화:", namgu_std.shape)
display(namgu_std.head())

print("\n북구 표준화:", bukgu_std.shape)
display(bukgu_std.head())

In [ ]:
# 저장된 14개 구·군 1차 정제 데이터를 다시 불러온 뒤,
# 남구·북구 공식 보완자료와 통합한다.

api_path = "/content/drive/MyDrive/부산_BigData_공모전/02_processed_data/유휴공유재산_1차정제.csv"

df_clean = pd.read_csv(api_path)

api_std = df_clean.copy()
api_std["data_source"] = "부산 구군 유휴 공유재산 API"

df_all = pd.concat(
    [api_std, namgu_std, bukgu_std],
    ignore_index=True
)

print("기존 API 데이터:", df_clean.shape)
print("통합 데이터:", df_all.shape)
print("구·군 수:", df_all["gugun"].nunique())

print("\n구·군별 건수")
print(
    df_all["gugun"]
    .value_counts()
    .sort_index()
)

In [ ]:
# 16개 구·군 통합 데이터의 핵심 품질을 최종 점검한다.
# 결측·중복·면적 이상값을 확인하되 아직 행을 삭제하지 않는다.

print("=== 기본 구조 ===")
print("행/열:", df_all.shape)
print("구·군 수:", df_all["gugun"].nunique())

print("\n=== 핵심 변수 결측 ===")
check_cols = ["gugun", "lctnAddr", "zoneArea", "land_type", "data_source"]
print(df_all[check_cols].isna().sum())

print("\n=== 주소 공백 ===")
blank_addr = df_all["lctnAddr"].astype(str).str.strip().eq("").sum()
print("주소 공백:", blank_addr)

print("\n=== 면적 이상값 ===")
print("면적 0:", (df_all["zoneArea"] == 0).sum())
print("면적 음수:", (df_all["zoneArea"] < 0).sum())
print("면적 결측:", df_all["zoneArea"].isna().sum())

print("\n=== 완전 중복 ===")
print("완전 중복:", df_all.duplicated().sum())

print("\n=== 주소 기준 중복 ===")
addr_dup = df_all.duplicated(
    subset=["gugun", "lctnAddr"],
    keep=False
).sum()
print("동일 구·군 내 주소 중복 행:", addr_dup)

print("\n=== 출처별 건수 ===")
print(df_all["data_source"].value_counts(dropna=False))

## 16개 구·군 유휴 공유재산 데이터 통합 결과

유휴 공유재산 API에서 NODATA가 발생한 남구와 북구를
각 구의 2026년 공식 공유재산 현황 자료로 보완하였다.

### 통합 결과

- 기존 API 자료: 9,404건 / 14개 구·군
- 남구 공식 보완자료: 651건
- 북구 공식 보완자료: 47건
- 최종 통합자료: 10,102건 / 16개 구·군

### 품질 검증 결과

- 주소 결측 및 공백: 0건
- 면적 결측: 0건
- 음수 면적: 0건
- 완전 중복: 0건
- 동일 구·군 내 주소 중복: 0건
- 면적 0㎡: 1건
- 지목 결측: 651건

지목 결측 651건은 모두 남구 공식자료에서 지목 정보를 제공하지 않아 발생한
구조적 결측으로, 임의로 값을 추정하거나 해당 행을 삭제하지 않는다.

면적 0㎡ 1건 역시 원자료의 값을 임의로 수정하지 않고 유지하며,
향후 활용유형별 유휴재산 선별 단계에서 적합성 검토 대상으로 처리한다.

### 분석 적용

최종 통합자료는 부산 16개 구·군의 유휴 공유재산 분석에 활용한다.

다만 자료의 출처와 기준시점이 서로 다르므로 이를 명시한다.

- 14개 구·군: 부산 구·군 유휴 공유재산 API
- 남구: 공유재산 현황 (2026.02.11)
- 북구: 공유재산 현황 (2026.06.30)

향후 생활수요 및 공공서비스 공급 분석을 통해 활용유형을 결정한 후,
각 활용유형에 필요한 면적·지목 등의 조건을 설정하여 유휴 공유재산과 매칭한다.

In [ ]:
# 검증이 완료된 16개 구·군 통합 데이터를 별도 파일로 저장한다.
# 기존 14개 구·군 1차 정제 파일은 원본 추적을 위해 유지한다.

save_path = (
    "/content/drive/MyDrive/부산_BigData_공모전/"
    "02_processed_data/유휴공유재산_16개구군_통합.csv"
)

df_all.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", save_path)
print("저장 데이터:", df_all.shape)